# 04 - Baseline models
Stratified 80/20 split (`random_state=42`), then **5-fold stratified cross-validation on the training data only** for the four baselines. The test set is used once, at the end, for reporting.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
import numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from src import config
config.ensure_directories()

In [2]:
from src.data.load_data import load_raw_data
from src.data.clean_data import clean_raw_dataframe, split_features_target
from src.models import model_training as mt
from src.models.baseline_models import get_baseline_models
from src.evaluation.evaluate_models import evaluate_on_test
df, _ = clean_raw_dataframe(load_raw_data())
X, y, ids = split_features_target(df)
X_train, X_test, y_train, y_test, ids_train, ids_test = mt.split_data(X, y, ids)
print('train', X_train.shape, f'churn={y_train.mean():.3f}', '| test', X_test.shape, f'churn={y_test.mean():.3f}')

train (5634, 19) churn=0.265 | test (1409, 19) churn=0.265


In [3]:
baselines = {k: mt.make_pipeline(est) for k, est in get_baseline_models(y_train).items()}
for k, p in baselines.items():
    print(k, '->', p.named_steps['model'])

logistic_regression -> LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
decision_tree -> DecisionTreeClassifier(class_weight='balanced', max_depth=5,
                       min_samples_leaf=20, random_state=42)
random_forest -> RandomForestClassifier(class_weight='balanced_subsample', max_depth=10,
                       min_samples_leaf=5, n_estimators=300, n_jobs=-1,
                       random_state=42)
xgboost -> XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_ste

In [4]:
cv_folds, cv_summary = mt.cross_validate_models(baselines, X_train, y_train)
cols = ['Model'] + [f'{m} {s}' for m in ('ROC-AUC', 'PR-AUC', 'F1', 'Recall', 'Specificity') for s in ('Mean', 'Std')]
cv_summary[cols].round(4)

,Model,ROC-AUC Mean,ROC-AUC Std,PR-AUC Mean,PR-AUC Std,F1 Mean,F1 Std,Recall Mean,Recall Std,Specificity Mean,Specificity Std
0,Logistic Regression,0.8449,0.0133,0.6610,0.0139,0.6259,0.0231,0.7933,0.0384,0.7323,0.0147
1,Decision Tree,0.8318,0.0138,0.6176,0.0306,0.6179,0.0098,0.7659,0.0363,0.7427,0.0190
2,Random Forest,0.8459,0.0102,0.6590,0.0240,0.6319,0.0157,0.7385,0.0256,0.7838,0.0095
3,XGBoost,0.8402,0.0102,0.6520,0.0217,0.6280,0.0144,0.7585,0.0244,0.7627,0.0081


In [5]:
fitted = mt.fit_pipelines(baselines, X_train, y_train)
comparison, predictions, confusion = evaluate_on_test(fitted, X_test, y_test, ids_test)
comparison.round(4)

,Model,Accuracy,Precision,Recall,Sensitivity,Specificity,F1,ROC-AUC,PR-AUC,MCC,Cohen's Kappa,Log Loss
0,Logistic Regression,0.7353,0.5008,0.7888,0.7888,0.7159,0.6127,0.8420,0.6370,0.4518,0.4264,0.4950
1,Decision Tree,0.7488,0.5181,0.7674,0.7674,0.7420,0.6185,0.8321,0.6244,0.4605,0.4415,0.5673
2,Random Forest,0.7693,0.5476,0.7540,0.7540,0.7749,0.6344,0.8413,0.6462,0.4850,0.4721,0.4623
3,XGBoost,0.7559,0.5271,0.7807,0.7807,0.7469,0.6293,0.8422,0.6551,0.4770,0.4573,0.4774


In [6]:
pd.DataFrame(confusion).T

,TN,FP,FN,TP
Logistic Regression,741,294,79,295
Decision Tree,768,267,87,287
Random Forest,802,233,92,282
XGBoost,773,262,82,292


**Reading the table** - positive class = churn. Recall/Sensitivity = share of real churners caught; Specificity = share of real non-churners correctly identified. Accuracy alone would be misleading because only ~26 % of customers churn.